In [0]:
# Retrieve the task value from the previous task (bronze)
bronze_output = dbutils.jobs.taskValues.get(taskKey="Bronze", key="bronze_output")

# Access individual variables
start_date = bronze_output.get("start_date", "")
bronze_adls = bronze_output.get("bronze_adls", "")
silver_adls = bronze_output.get("silver_adls", "")

print(f"Start Date: {start_date}, Bronze ADLS: {bronze_adls}")

Start Date: 2026-02-21, Bronze ADLS: abfss://bronze@sdfgsdfgs456ghh.dfs.core.windows.net/


In [0]:
from pyspark.sql.functions import col, isnull, when
from pyspark.sql.types import TimestampType
from datetime import date, timedelta
     

In [0]:

# Load the JSON data into a Spark DataFrame
df = spark.read.option("multiline", "true").json(f"{bronze_adls}{start_date}_earthquake_data.json")
     

In [0]:

df.head()

Row(geometry=Row(coordinates=[-66.7823333333333, 18.0391666666667, 15.6], type='Point'), id='pr71508438', properties=Row(alert=None, cdi=None, code='71508438', detail='https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=pr71508438&format=geojson', dmin=0.1054, felt=None, gap=143, ids=',pr71508438,', mag=2.07, magType='md', mmi=None, net='pr', nst=7, place='2 km NNE of Guayanilla, Puerto Rico', rms=0.1, sig=66, sources=',pr,', status='reviewed', time=1771718113570, title='M 2.1 - 2 km NNE of Guayanilla, Puerto Rico', tsunami=0, type='earthquake', types=',origin,phase-data,', tz=None, updated=1771720386560, url='https://earthquake.usgs.gov/earthquakes/eventpage/pr71508438'), type='Feature')

In [0]:
# Reshape earthquake data
df = (
    df
    .select(
        'id',
        col('geometry.coordinates').getItem(0).alias('longitude'),
        col('geometry.coordinates').getItem(1).alias('latitude'),
        col('geometry.coordinates').getItem(2).alias('elevation'),
        col('properties.title').alias('title'),
        col('properties.place').alias('place_description'),
        col('properties.sig').alias('sig'),
        col('properties.mag').alias('mag'),
        col('properties.magType').alias('magType'),
        col('properties.time').alias('time'),
        col('properties.updated').alias('updated')
    )
)

In [0]:

df.head()

Row(id='pr71508438', longitude=-66.7823333333333, latitude=18.0391666666667, elevation=15.6, title='M 2.1 - 2 km NNE of Guayanilla, Puerto Rico', place_description='2 km NNE of Guayanilla, Puerto Rico', sig=66, mag=2.07, magType='md', time=1771718113570, updated=1771720386560)

In [0]:
# Validate data: Check for missing or null values
df = (
    df
    .withColumn('longitude', when(isnull(col('longitude')), 0).otherwise(col('longitude')))
    .withColumn('latitude', when(isnull(col('latitude')), 0).otherwise(col('latitude')))
    .withColumn('time', when(isnull(col('time')), 0).otherwise(col('time')))
)

In [0]:
# Convert 'time' and 'updated' to timestamp from Unix time
df = (
    df
    .withColumn('time', (col('time') / 1000).cast(TimestampType()))
    .withColumn('updated', (col('updated') / 1000).cast(TimestampType()))
)

In [0]:
df.head()

Row(id='pr71508438', longitude=-66.7823333333333, latitude=18.0391666666667, elevation=15.6, title='M 2.1 - 2 km NNE of Guayanilla, Puerto Rico', place_description='2 km NNE of Guayanilla, Puerto Rico', sig=66, mag=2.07, magType='md', time=datetime.datetime(2026, 2, 21, 23, 55, 13, 570000), updated=datetime.datetime(2026, 2, 22, 0, 33, 6, 560000))

In [0]:
# Save the transformed DataFrame to the Silver container
silver_output_path = f"{silver_adls}earthquake_events_silver/"

In [0]:
# Append DataFrame to Silver container in Parquet format
df.write.mode('append').parquet(silver_output_path)

In [0]:
dbutils.jobs.taskValues.set(key = "silver_output", value = silver_output_path)